# Distillation System Identification

This notebook is the canonical distillation system-identification workflow. It writes the canonical model, scaling, and min/max files into `Distillation/Data`, while preserving the general tone and step-by-step structure of the archived distillation system-identification work.

In [ ]:
from pathlib import Path
import os
import pickle

from systems.distillation import get_distillation_notebook_defaults
from utils.notebook_setup import prepare_distillation_notebook_env, print_notebook_summary

NB = get_distillation_notebook_defaults("system_identification")
RUN_NEW_EXPERIMENTS = NB["run_new_experiments"]
USE_RHP_ZERO = NB["use_rhp_zero"]
SHOW_FOPDT_PLOTS = NB["show_fopdt_plots"]
SHOW_VALIDATION_PLOTS = NB["show_validation_plots"]
ASPEN_PRESET = NB["aspen_preset"]
ASPEN_PATH_OVERRIDE = NB["aspen_path_override"]
SNAPS_PATH_OVERRIDE = NB["snaps_path_override"]
ASPEN_ROOT_OVERRIDE = NB["aspen_root_override"]
DISTILLATION_VISIBLE = NB["distillation_visible"]
DISTILLATION_DATA_DIR_OVERRIDE = NB["data_dir_override"]
DISTILLATION_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]

REPO_ROOT, DATA_DIR, RESULT_DIR, DISTURBANCE_PROFILE, DYN_PATH, SNAPS_PATH, ASPEN_SOURCE = prepare_distillation_notebook_env(run_mode="nominal", disturbance_profile="none", family="system_id", aspen_preset=ASPEN_PRESET, dyn_path_override=ASPEN_PATH_OVERRIDE, snaps_path_override=SNAPS_PATH_OVERRIDE, aspen_root_override=ASPEN_ROOT_OVERRIDE, data_dir_override=DISTILLATION_DATA_DIR_OVERRIDE, results_dir_override=DISTILLATION_RESULTS_DIR_OVERRIDE)
os.chdir(REPO_ROOT)
CARRY_FORWARD_MIN_MAX = DATA_DIR / NB["carry_forward_min_max_name"]


In [ ]:
import numpy as np
import pandas as pd

from systems.distillation.config import (
    DELTA_T_HOURS,
    DISTILLATION_NOMINAL_CONDITIONS,
    DISTILLATION_SS_INPUTS_SYSTEM_ID,
)
from systems.distillation.data_io import resolve_distillation_data_dir
from systems.distillation.plant import DistillationColumnAspen, build_distillation_system
from systems.distillation.system_id import (
    apply_deviation_form_scaled,
    build_delay_list_hours,
    extract_fopdt_2863,
    plot_state_space_validation,
    run_distillation_experiment,
    save_canonical_system_identification,
    scaling_min_max_factors,
    state_space_form_using_matlab,
)


In [ ]:
SYS = NB["system_setup"]
DATA_DIR = resolve_distillation_data_dir(REPO_ROOT, override=DISTILLATION_DATA_DIR_OVERRIDE)
CARRY_FORWARD_MIN_MAX = DATA_DIR / NB["carry_forward_min_max_name"]
nominal_conditions = SYS["nominal_conditions"].copy()
ss_inputs = SYS["ss_inputs_system_id"].copy()
print(f"Canonical distillation data dir: {DATA_DIR}")
print(f"Using Aspen dynf: {DYN_PATH}")
print(f"Using Aspen snaps: {SNAPS_PATH}")


In [ ]:
steady_states = {"ss_inputs": ss_inputs.copy(), "y_ss": None}

if RUN_NEW_EXPERIMENTS:
    system = build_distillation_system(path=DYN_PATH, ss_inputs=ss_inputs, initialization_point=nominal_conditions, delta_t=SYS["delta_t_hours"], visible=DISTILLATION_VISIBLE)
    try:
        steady_states = {"ss_inputs": np.asarray(system.ss_inputs, float).copy(), "y_ss": np.asarray(system.y_ss, float).copy()}
    finally:
        try:
            system.close(SNAPS_PATH)
        except Exception:
            pass

    for step_cfg in NB["step_tests"]:
        run_distillation_experiment(
            system_cls=DistillationColumnAspen,
            path=DYN_PATH,
            ss_inputs=steady_states["ss_inputs"],
            nominal_conditions=nominal_conditions,
            delta_t=SYS["delta_t_hours"],
            step_value=step_cfg["step_value"],
            step_channel=step_cfg["step_channel"],
            save_filename=step_cfg["save_filename"],
            data_dir=DATA_DIR,
        )
else:
    print("RUN_NEW_EXPERIMENTS=False, using the canonical/archived distillation data already present in Distillation/Data.")

print(steady_states)


In [ ]:
file_paths = {
    "Reflux": DATA_DIR / "Reflux.csv",
    "Reboiler": DATA_DIR / "Reboiler.csv",
}

data_min, data_max = scaling_min_max_factors(file_paths)
scaling_factor = {
    "min": np.asarray(data_min, float),
    "max": np.asarray(data_max, float),
}

with open(DATA_DIR / "system_dict.pickle", "rb") as handle:
    legacy_system_dict = pickle.load(handle)
with open(CARRY_FORWARD_MIN_MAX, "rb") as handle:
    legacy_min_max_states = pickle.load(handle)

if RUN_NEW_EXPERIMENTS:
    if steady_states["y_ss"] is None:
        raise RuntimeError("Steady-state outputs are required when regenerating the distillation identification assets.")
    deviation_dfs = apply_deviation_form_scaled(steady_states, file_paths, data_min, data_max)
else:
    deviation_dfs = {name: pd.read_csv(path) for name, path in file_paths.items()}

print("Scaling min:", scaling_factor["min"])
print("Scaling max:", scaling_factor["max"])


In [ ]:
if RUN_NEW_EXPERIMENTS:
    reflux_dict, reflux_data = extract_fopdt_2863(
        deviation_dfs["Reflux"],
        input_idx=0,
        Ts=DELTA_T_HOURS,
        limit=25,
        pre_win=30,
        post_win=30,
        plot=SHOW_FOPDT_PLOTS,
    )
    reboiler_dict, reboiler_data = extract_fopdt_2863(
        deviation_dfs["Reboiler"],
        input_idx=1,
        Ts=DELTA_T_HOURS,
        limit=500,
        pre_win=30,
        post_win=500,
        plot=SHOW_FOPDT_PLOTS,
    )

    delay_list_hours, delay_samples = build_delay_list_hours(
        reflux_dict,
        reboiler_dict,
        y1_name="Tray24_C2H6",
        y2_name="Tray85_T",
        Ts_hours=DELTA_T_HOURS,
        quantization="ceil",
    )

    A, B, C, D, y_step_sum, y_step, tOut = state_space_form_using_matlab(
        u1_dict=reflux_dict,
        u2_dict=reboiler_dict,
        delay_list_hours=delay_list_hours,
        data_u1=reflux_data,
        data_u2=reboiler_data,
        sampling_time=DELTA_T_HOURS,
        use_rhp_zero=USE_RHP_ZERO,
        return_step=True,
    )

    if SHOW_VALIDATION_PLOTS:
        plot_state_space_validation(tOut, y_step, reflux_data, reboiler_data, Ts=DELTA_T_HOURS)

    system_dict = {"A": A, "B": B, "C": C, "D": D}
else:
    system_dict = legacy_system_dict

system_dict


In [ ]:
save_canonical_system_identification(
    data_dir=DATA_DIR,
    system_dict=system_dict,
    scaling_factor=scaling_factor,
    min_max_states=legacy_min_max_states,
)

print(f"Saved canonical distillation identification assets to {DATA_DIR}")
print(sorted(p.name for p in DATA_DIR.glob("*.pickle")))
